⚠️ This project is mandatory for certification bloc #2.

![Tinder](https://full-stack-assets.s3.eu-west-3.amazonaws.com/M03-EDA/Tinder-Symbole.png)

# Speed Dating with Tinder

## Company's description 📇

<a href="https://tinder.com/" target="_blank">Tinder</a> is an online dating and geosocial networking application. In Tinder, users "swipe right" to like or "swipe left" to dislike other users' profiles, which include their photos, a short bio, and a list of their interests.

Tinder was launched by Sean Rad at a hackathon held at the Hatch Labs incubator in West Hollywood in 2012.

As of 2021, Tinder has recorded more than 65 billion matches worldwide.

## Project 🚧

The marketing team needs help on a new project. They are experiencing a decrease in the number of matches, and they are trying to find a way to understand **what makes people interested into each other**. 

They decided to run a speed dating experiment with people who had to give Tinder lots of informations about themselves that could ultimately reflect on ther dating profile on the app.

Tinder then gathered the data from this experiment. Each row in the dataset represents one speed date between two people, and indicates wether each of them secretly agreed to go on a second date with the other person.

## Goals 🎯

Use the dataset to understand what makes people interested into each other to go on a second date together:
* You may use descriptive statistics
* You may use visualisations

## Scope of this project 🖼️

Data was gathered from participants in experimental speed dating events from 2002-2004. During the events, the attendees would have a four minute "first date" with every other participant of the opposite sex. At the end of their four minutes, participants were asked if they would like to see their date again. They were also asked to rate their date on six attributes: Attractiveness, Sincerity, Intelligence, Fun, Ambition, and Shared Interests.

The dataset also includes questionnaire data gathered from participants at different points in the process. These fields include: demographics, dating habits, self-perception across key attributes, beliefs on what others find valuable in a mate, and lifestyle information. See the Speed Dating Data Key document below for details.

[Dataset](https://full-stack-assets.s3.eu-west-3.amazonaws.com/M03-EDA/Speed+Dating+Data.csv)

[Dataset Description](https://full-stack-assets.s3.eu-west-3.amazonaws.com/M03-EDA/Speed+Dating+Data+Key.doc)

## Helpers 🦮

To help you achieve this project, here are a few tips that should help you

Data Exploration Ideas :
* What are the least desirable attributes in a male partner? Does this differ for female partners?
* How important do people think attractiveness is in potential mate selection vs. its real impact?
* Are shared interests more important than a shared racial background?
* Can people accurately predict their own perceived value in the dating market?
* In terms of getting a second date, is it better to be someone's first speed date of the night or their last?

## Deliverable 📬

To complete this project, your team should deliver:

A notebook with:
* descriptive statistics
* visualisations
* captions and interpretations on how the stats and visualisations are relevant to why people agree to a second date

# 💘 Projet Speed Dating : qu'est-ce qui donne envie de se revoir ?

**Contexte :** L'équipe marketing de Tinder constate une baisse du nombre de mises en
relation. Elle veut comprendre ce qui donne envie à deux personnes de se revoir.

**Données :** une expérience de speed dating (2002-2004). Chaque ligne est un rendez-vous
de 4 minutes entre deux personnes. Chacune dit "Oui" ou "Non" à un second rendez-vous
et note l'autre sur 6 critères : attirance, sincérité, intelligence, humour, ambition,
intérêts communs.

**Ce que je fais :** contrôler et nettoyer les données, les analyser avec des statistiques
descriptives et des graphiques, puis formuler des recommandations pour Tinder. Je termine
par un calcul en parallèle avec PySpark.

## Imports et chargement

In [1]:
# 1. Importation des outils de travail
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# 2. Ouverture du fichier de données
df_tinder = pd.read_csv('../data/speed_dating.csv', encoding='latin-1')


# 3. Premier audit visuel de la taille du carton
lignes = df_tinder.shape[0]
colonnes = df_tinder.shape[1]
print(f"Audit initial : La base contient {lignes} rencontres et {colonnes} colonnes.")

# 4. On regarde à quoi ressemblent les 5 premières rencontres
display(df_tinder.head())

Audit initial : La base contient 8378 rencontres et 195 colonnes.


,iid,id,gender,idg,condtn,wave,round,position,positin1,order,...,attr3_3,sinc3_3,intel3_3,fun3_3,amb3_3,attr5_3,sinc5_3,intel5_3,fun5_3,amb5_3
0,1,1.0,0,1,1,1,10,7,NaN,4,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
1,1,1.0,0,1,1,1,10,7,NaN,3,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
2,1,1.0,0,1,1,1,10,7,NaN,10,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
3,1,1.0,0,1,1,1,10,7,NaN,5,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
4,1,1.0,0,1,1,1,10,7,NaN,7,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN


## Étape 1 : Où sont les cases vides ?

**Pourquoi cette étape ?** Avant d'analyser, je dois savoir si des données manquent. Si je supprime les lignes incomplètes, je risque de changer le profil des participants (par exemple avoir moins d'hommes ou moins de "Oui"). Je compare donc **avant / après** pour vérifier que ma base reste équilibrée.

In [2]:
# 1. Les 10 colonnes que je garde pour l'analyse (les mêmes que dans le nettoyage)
colonnes_analyse = ['iid', 'pid', 'gender', 'dec', 'attr', 'sinc', 'intel', 'fun', 'amb', 'shar']

# 2. Combien de cases vides dans chaque colonne ?
cases_vides = df_tinder[colonnes_analyse].isna().sum()
pourcentage_vides = (cases_vides / len(df_tinder) * 100).round(1)

tableau_vides = pd.DataFrame({'Cases vides': cases_vides, '% de vides': pourcentage_vides})
tableau_vides = tableau_vides.sort_values('% de vides', ascending=False)
display(tableau_vides)

# 3. Je simule la suppression des lignes qui ont au moins une case vide
base_sans_vides = df_tinder[colonnes_analyse].dropna()

# 4. Combien de lignes je perds ?
avant = len(df_tinder)
apres = len(base_sans_vides)
print("Lignes avant :", avant)
print("Lignes après :", apres)
print("Part perdue :", round((avant - apres) / avant * 100, 1), "%")

# 5. Est-ce que ça change mon échantillon ?
print("Part de 'Oui' avant :", round(df_tinder['dec'].mean() * 100, 1), "%")
print("Part de 'Oui' après :", round(base_sans_vides['dec'].mean() * 100, 1), "%")
print("Part d'hommes avant :", round(df_tinder['gender'].mean() * 100, 1), "%")
print("Part d'hommes après :", round(base_sans_vides['gender'].mean() * 100, 1), "%")

,Cases vides,% de vides
shar,1067,12.7
amb,712,8.5
fun,350,4.2
intel,296,3.5
sinc,277,3.3
attr,202,2.4
pid,10,0.1
iid,0,0.0
gender,0,0.0
dec,0,0.0


Lignes avant : 8378
Lignes après : 7031
Part perdue : 16.1 %
Part de 'Oui' avant : 42.0 %
Part de 'Oui' après : 43.2 %
Part d'hommes avant : 50.1 %
Part d'hommes après : 50.8 %


**Ce que je vois :** la base passe de 8378 à 7031 lignes (16,1 % perdus). La part de
« Oui » passe de 42,0 % à 43,2 % et la part d'hommes de 50,1 % à 50,8 %.

**Ce que je conclus :** les pourcentages bougent de moins de 2 points, donc l'échantillon
reste équilibré. Je peux supprimer les lignes incomplètes sans fausser l'analyse.

## Étape 2 : Nettoyage des données
Le fichier brut contient 195 colonnes. Je garde seulement les 10 qui servent à répondre à la question : les identifiants, le genre, la décision (Oui/Non) et les 6 notes. Je supprime ensuite les lignes où il manque une information, pour que les calculs soient justes. Il reste 7031 rencontres complètes sur 8378.

In [3]:
# 1. Sélection stricte (On garde uniquement nos 10 colonnes vitales, on ignore les 185 autres)
colonnes_a_garder = ['iid', 'pid', 'gender', 'dec', 'attr', 'sinc', 'intel', 'fun', 'amb', 'shar']
df_propre = df_tinder[colonnes_a_garder].copy()

# 2. Traduction Business
df_propre = df_propre.rename(columns={
    'iid': 'id_participant',
    'pid': 'id_partenaire',
    'gender': 'genre',
    'dec': 'decision_match',
    'attr': 'note_attractivite',
    'sinc': 'note_sincerite',
    'intel': 'note_intelligence',
    'fun': 'note_humour',
    'amb': 'note_ambition',
    'shar': 'note_interets_communs'
})

# 3. Nettoyage des NaN
df_propre = df_propre.dropna()

# 4. Pour la lisibilité des graphiques : on transforme les 0 et 1 du genre en mots
df_propre['genre'] = df_propre['genre'].replace({0.0: 'Femme', 1.0: 'Homme'})

# 5. Audit final de la base saine
lignes_propres = df_propre.shape[0]
print(f"Base nettoyée : Il reste {lignes_propres} rencontres 100% complètes et 10 colonnes.")
display(df_propre.head())

Base nettoyée : Il reste 7031 rencontres 100% complètes et 10 colonnes.


,id_participant,id_partenaire,genre,decision_match,note_attractivite,note_sincerite,note_intelligence,note_humour,note_ambition,note_interets_communs
0,1,11.0,Femme,1,6.0,9.0,7.0,7.0,6.0,5.0
1,1,12.0,Femme,1,7.0,8.0,7.0,8.0,5.0,6.0
2,1,13.0,Femme,1,5.0,8.0,9.0,8.0,5.0,7.0
3,1,14.0,Femme,1,7.0,6.0,8.0,7.0,6.0,8.0
4,1,15.0,Femme,1,5.0,6.0,7.0,7.0,6.0,6.0


## Étape 3 : Les notes et les âges sont-ils réalistes ?

**Pourquoi cette étape ?** Les notes vont de 0 à 10. Une note de 15 ou de -2 serait une erreur de saisie. Je regarde donc la plus petite (min) et la plus grande (max) valeur de chaque colonne. Je fais pareil pour l'âge.

In [4]:
# 1. Les 6 notes données pendant les rencontres
# (attr = attirance, sinc = sincérité, intel = intelligence,
#  fun = humour, amb = ambition, shar = intérêts communs)
notes = ['attr', 'sinc', 'intel', 'fun', 'amb', 'shar']

# 2. Résumé chiffré : min = plus petite note, max = plus grande note
resume_notes = df_tinder[notes].describe().round(1)
display(resume_notes)

# 3. Combien de notes sont hors de l'échelle 0 à 10 ?
for note in notes:
    hors_echelle = ((df_tinder[note] < 0) | (df_tinder[note] > 10)).sum()
    print(note, ": notes hors échelle =", hors_echelle)

# 4. Les âges des participants
display(df_tinder['age'].describe().round(1))

,attr,sinc,intel,fun,amb,shar
count,8176.0,8101.0,8082.0,8028.0,7666.0,7311.0
mean,6.2,7.2,7.4,6.4,6.8,5.5
std,2.0,1.7,1.6,2.0,1.8,2.2
min,0.0,0.0,0.0,0.0,0.0,0.0
25%,5.0,6.0,6.0,5.0,6.0,4.0
50%,6.0,7.0,7.0,7.0,7.0,6.0
75%,8.0,8.0,8.0,8.0,8.0,7.0
max,10.0,10.0,10.0,10.0,10.0,10.0


attr : notes hors échelle = 0
sinc : notes hors échelle = 0
intel : notes hors échelle = 0
fun : notes hors échelle = 0
amb : notes hors échelle = 0
shar : notes hors échelle = 0


count    8283.0
mean       26.4
std         3.6
min        18.0
25%        24.0
50%        26.0
75%        28.0
max        55.0
Name: age, dtype: float64

## Étape 4 : Quels critères sont les plus liés au "Oui" ?

**Pourquoi cette étape ?** Pour conseiller Tinder, je cherche quelle note va le plus avec la décision "Oui". J'utilise la corrélation de Pearson : un chiffre entre -1 et 1. Proche de 1, la note et le "Oui" montent ensemble. Proche de 0, il n'y a pas de lien.

In [5]:
# 1. Je garde la décision et les 6 notes, sans cases vides
df_corr = df_tinder[['dec', 'attr', 'sinc', 'intel', 'fun', 'amb', 'shar']].dropna()

# 2. Je calcule le lien de chaque note avec la décision (corrélation de Pearson)
lien_decision = df_corr.corr(method='pearson')['dec'].drop('dec').round(2)

# 3. Je mets des noms lisibles
lien_decision = lien_decision.rename({'attr': 'Attirance', 'sinc': 'Sincérité',
                                      'intel': 'Intelligence', 'fun': 'Humour',
                                      'amb': 'Ambition', 'shar': 'Intérêts communs'})

# 4. Je prépare le tableau pour le graphique, du plus fort au plus faible
tableau_lien = lien_decision.reset_index()
tableau_lien.columns = ['Critère', 'Lien avec le Oui']
tableau_lien = tableau_lien.sort_values('Lien avec le Oui', ascending=False)
display(tableau_lien)

# 5. Graphique en barres (l'axe va de 0 à 1 pour ne pas exagérer les écarts)
fig = px.bar(tableau_lien, x='Critère', y='Lien avec le Oui', text_auto=True,
             title="Quels critères sont les plus liés à la décision 'Oui' ?")
fig.update_yaxes(range=[0, 1])
fig.show()

,Critère,Lien avec le Oui
0,Attirance,0.49
3,Humour,0.41
5,Intérêts communs,0.40
1,Sincérité,0.21
2,Intelligence,0.21
4,Ambition,0.18


## Étape 5 : L'attirance compte-t-elle dans le "Oui" ?
Je compare la note d'attirance moyenne quand la réponse est "Non" et quand elle est "Oui", séparément pour les hommes et les femmes.

In [6]:
# 1. Nettoyage du texte pour l'axe (Oui / Non)
df_propre['decision_match'] = df_propre['decision_match'].astype(str).replace({'0.0': 'Non', '1.0': 'Oui', '0': 'Non', '1': 'Oui'})

# 2. Agrégation : On calcule la moyenne exacte pour simplifier la lecture
df_moyenne = df_propre.groupby(['decision_match', 'genre'])['note_attractivite'].mean().reset_index()

# 3. Création d'un graphique en barres clair et universel
fig_beaute_simple = px.bar(
    df_moyenne,
    x="decision_match",
    y="note_attractivite",
    color="genre",
    barmode="group", # Met les barres côte à côte au lieu de les empiler
    text_auto='.1f', # Affiche le chiffre exact sur la barre
    title="L'impact du physique sur le Match (Note Moyenne)",
    labels={
        "decision_match": "Décision (Voulez-vous revoir la personne ?)",
        "note_attractivite": "Note moyenne de beauté (sur 10)",
        "genre": "Genre"
    },
    color_discrete_map={"Femme": "#EF553B", "Homme": "#636EFA"}
)

# 4. Épuration du design
fig_beaute_simple.update_layout(template="plotly_white")
fig_beaute_simple.show()

# 5. Les chiffres exacts, pour pouvoir les citer
display(df_moyenne.round(1))

,decision_match,genre,note_attractivite
0,Non,Femme,5.2
1,Non,Homme,5.5
2,Oui,Femme,7.1
3,Oui,Homme,7.4


## Étape 6 : Qu'est-ce qui fait dire "Oui" ? (écart Oui / Non)

**Pourquoi cette étape ?** Pour chaque critère, je compare la note moyenne quand la réponse est "Oui" et quand elle est "Non". L'**écart** (note "Oui" moins note "Non") montre quels critères font vraiment la différence. Un grand écart positif : le critère fait dire "Oui". Un écart proche de 0 : il compte peu. Je sépare les hommes et les femmes (genre de la personne qui note).

In [7]:
# 1. Les 6 notes (avec des noms lisibles pour le graphique)
criteres = ['note_attractivite', 'note_sincerite', 'note_intelligence',
            'note_humour', 'note_ambition', 'note_interets_communs']
noms_lisibles = {'note_attractivite': 'Attirance', 'note_sincerite': 'Sincérité',
                 'note_intelligence': 'Intelligence', 'note_humour': 'Humour',
                 'note_ambition': 'Ambition', 'note_interets_communs': 'Intérêts communs'}

# 2. Note moyenne de chaque critère, selon le genre et la décision
moyennes = df_propre.groupby(['genre', 'decision_match'])[criteres].mean()

# 3. Écart = note moyenne quand c'est "Oui" moins note moyenne quand c'est "Non"
moyenne_oui = moyennes.xs('Oui', level='decision_match')
moyenne_non = moyennes.xs('Non', level='decision_match')
ecarts = (moyenne_oui - moyenne_non).round(2).rename(columns=noms_lisibles)
display(ecarts)

# 4. Je prépare le tableau pour le graphique
df_ecarts = ecarts.reset_index().melt(id_vars='genre', var_name='Critère',
                                      value_name='Écart Oui - Non')

# 5. Graphique en barres horizontales
fig = px.bar(df_ecarts, x='Écart Oui - Non', y='Critère', color='genre',
             barmode='group', orientation='h', text_auto='.1f',
             title="Qu'est-ce qui fait dire Oui ? (note quand Oui moins note quand Non)",
             color_discrete_map={"Femme": "#EF553B", "Homme": "#636EFA"})
fig.update_layout(template="plotly_white")
fig.show()

,Attirance,Sincérité,Intelligence,Humour,Ambition,Intérêts communs
genre,,,,,,
Femme,1.84,0.86,0.79,1.78,0.70,1.88
Homme,1.92,0.60,0.63,1.49,0.74,1.63


## Étape 7 : Premier ou dernier rendez-vous de la soirée ?
La position dans la soirée change-t-elle la chance d'un "Oui" ? Je regarde d'abord les 10 premières positions, puis 3 groupes : premier, milieu, dernier.

In [8]:
# 1. On récupère les colonnes "Ordre de passage" et "Décision" depuis le fichier brut
df_chrono = df_tinder[['order', 'dec']].dropna()

# 2. Statistiques descriptives : On calcule la moyenne de "Oui" pour chaque position
df_chrono_moy = df_chrono.groupby('order')['dec'].mean().reset_index()

# 3. On multiplie par 100 pour transformer les 0.40 en 40% (plus lisible)
df_chrono_moy['dec'] = df_chrono_moy['dec'] * 100

# 4. On coupe à 10 dates pour garder le graphique lisible
df_chrono_moy = df_chrono_moy[df_chrono_moy['order'] <= 10]

# 5. Visualisation : Graphique en barres simple
fig_chrono = px.bar(
    df_chrono_moy,
    x='order',
    y='dec',
    text_auto='.0f', # Affiche le pourcentage entier sur la barre
    title="Taux de Oui selon la position dans la soirée (10 premières positions)",
    labels={'order': "Position dans la soirée (1 = 1er date)", 'dec': "Taux de Oui (%)"},
    color_discrete_sequence=['#FFB6C1']
)

fig_chrono.update_layout(template="plotly_white")
fig_chrono.show()

In [9]:
# 1. Vérification : l'ordre ne doit jamais dépasser le nombre de rencontres de la soirée
print("Lignes où order > round (doit être 0) :", (df_tinder['order'] > df_tinder['round']).sum())

# 2. Je garde les colonnes utiles
df_dernier = df_tinder[['order', 'round', 'dec']].dropna().copy()

# 3. Je range chaque rencontre dans un groupe
df_dernier['Position'] = 'Milieu'
df_dernier.loc[df_dernier['order'] == 1, 'Position'] = 'Premier'
df_dernier.loc[df_dernier['order'] == df_dernier['round'], 'Position'] = 'Dernier'

# 4. Taux de "Oui" par groupe
taux_position = (df_dernier.groupby('Position')['dec'].mean() * 100).round(1).reset_index()
taux_position.columns = ['Position', 'Taux de Oui (%)']
display(taux_position)

# 5. Graphique
fig = px.bar(taux_position, x='Position', y='Taux de Oui (%)', text_auto=True,
             category_orders={'Position': ['Premier', 'Milieu', 'Dernier']},
             title="Taux de Oui : premier, milieu ou dernier rendez-vous")
fig.update_yaxes(range=[0, 100])
fig.show()

Lignes où order > round (doit être 0) : 0


,Position,Taux de Oui (%)
0,Dernier,44.5
1,Milieu,41.2
2,Premier,49.9


## Étape 8 : Intérêts communs ou même origine ?
Les intérêts communs comptent-ils plus que d'avoir la même origine ? Je compare le taux de "Oui" selon que les partenaires ont la même origine ou non, puis selon que la note d'intérêts communs est forte (7 ou plus) ou faible (moins de 7).

In [10]:
# 1. On récupère les données utiles : Décision, Origine identique (1=Oui, 0=Non), et Note d'intérêts (sur 10)
df_valeurs = df_tinder[['dec', 'samerace', 'shar']].dropna()

# 2. Catégorisation : On transforme la note mathématique en texte clair
df_valeurs['Meme_Origine'] = df_valeurs['samerace'].replace({1.0: 'Oui', 0.0: 'Non'})
df_valeurs['Interets_Communs'] = df_valeurs['shar'].apply(lambda x: 'Forts (Note >= 7)' if x >= 7 else 'Faibles (Note < 7)')

# 3. Statistiques Descriptives : Calcul des pourcentages de Match
tx_origine = df_valeurs.groupby('Meme_Origine')['dec'].mean() * 100
tx_interets = df_valeurs.groupby('Interets_Communs')['dec'].mean() * 100

# 4. Affichage du rapport textuel
print("--- STATISTIQUES DESCRIPTIVES : CE QUI DÉCLENCHE LE MATCH ---")
print(f"Match si les partenaires ont la MÊME ORIGINE : {tx_origine['Oui']:.1f} %")
print(f"Match si les partenaires ont une ORIGINE DIFFÉRENTE : {tx_origine['Non']:.1f} %")
print("-" * 50)
print(f"Match si les INTÉRÊTS COMMUNS sont FORTS : {tx_interets['Forts (Note >= 7)']:.1f} %")
print(f"Match si les INTÉRÊTS COMMUNS sont FAIBLES : {tx_interets['Faibles (Note < 7)']:.1f} %")

--- STATISTIQUES DESCRIPTIVES : CE QUI DÉCLENCHE LE MATCH ---
Match si les partenaires ont la MÊME ORIGINE : 45.0 %
Match si les partenaires ont une ORIGINE DIFFÉRENTE : 42.2 %
--------------------------------------------------
Match si les INTÉRÊTS COMMUNS sont FORTS : 66.8 %
Match si les INTÉRÊTS COMMUNS sont FAIBLES : 31.6 %


# Étape 9 : Calcul en parallèle avec PySpark

**Pourquoi ?** Tinder gère un très grand nombre de rencontres. Avec Pandas, tout se
calcule sur un seul "morceau". Avec Spark, les données sont coupées en plusieurs morceaux
traités en même temps.

**Ce que je fais :** je calcule le taux de "Oui" des hommes et des femmes avec Spark, puis
je compare avec Pandas pour vérifier que j'obtiens le même résultat.

**Limite :** ici, Spark tourne sur mon ordinateur (mode "local"), pas sur plusieurs
machines. Avec seulement quelques milliers de lignes, Pandas est aussi rapide. L'intérêt est
de montrer la méthode, qui fonctionne telle quelle sur un cluster avec des millions de lignes.

In [11]:
# 1. Je dis à Python où se trouve Java 17 (Spark en a besoin)
import os
import sys
chemin_java = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"
if os.path.exists(chemin_java):
    os.environ["JAVA_HOME"] = chemin_java
os.environ["PYSPARK_PYTHON"] = sys.executable

# 2. Importation des outils Spark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, round as arrondi

# 3. Démarrage de Spark sur mon ordinateur (local[*] = tous les coeurs du processeur)
spark = SparkSession.builder.master("local[*]").appName("Tinder_Speed_Dating").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# 4. Lecture du fichier (même encodage qu'avec Pandas)
df_spark = spark.read.csv("../data/speed_dating.csv", header=True, inferSchema=True, encoding="ISO-8859-1")
print("Nombre de lignes lues par Spark :", df_spark.count())

# 5. Je coupe les données en 4 morceaux traités en parallèle
df_spark = df_spark.repartition(4)
print("Nombre de morceaux traités en parallèle :", df_spark.rdd.getNumPartitions())

# 6. Calcul en parallèle : taux de "Oui" par genre
df_resultat = (df_spark.filter(col("dec").isNotNull())
               .groupBy("gender")
               .agg(arrondi(avg("dec") * 100, 1).alias("Taux_de_Oui_Pourcent"),
                    count("*").alias("Nb_rencontres")))
print("--- RÉSULTAT AVEC SPARK ---")
df_resultat.show()

# 7. Vérification : même calcul avec Pandas
print("--- MÊME CALCUL AVEC PANDAS ---")
print((df_tinder.groupby('gender')['dec'].mean() * 100).round(1))

# 8. J'éteins Spark
spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/27 13:52:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Nombre de lignes lues par Spark : 8378
Nombre de morceaux traités en parallèle : 4
--- RÉSULTAT AVEC SPARK ---
+------+--------------------+-------------+
|gender|Taux_de_Oui_Pourcent|Nb_rencontres|
+------+--------------------+-------------+
|     1|                47.4|         4194|
|     0|                36.5|         4184|
+------+--------------------+-------------+

--- MÊME CALCUL AVEC PANDAS ---
gender
0    36.5
1    47.4
Name: dec, dtype: float64


**Ce que je vois :** Spark lit 8378 rencontres, les coupe en 4 morceaux traités en
parallèle, et trouve un taux de "Oui" de 47,4 % pour les hommes et de 36,5 % pour les
femmes. Pandas donne exactement les mêmes chiffres : le calcul est donc vérifié.

**Ce que ça veut dire pour Tinder :** les hommes disent "Oui" plus souvent que les
femmes (environ 11 points d'écart). Les femmes sont plus sélectives.

**Limite :** Spark tourne ici sur un seul ordinateur. Avec si peu de lignes, il n'est pas
plus rapide que Pandas : je montre la méthode, qui fonctionnerait aussi avec des millions
de rencontres.

## Étape 10 : Conclusion et recommandations pour Tinder

**Ce que j'ai trouvé :**
- Le "Oui" est surtout lié à l'**attirance** (0,49), à l'**humour** (0,41) et aux **intérêts communs** (0,40). La sincérité, l'intelligence et l'ambition pèsent moins (0,18 à 0,21).
- Les **intérêts communs** comptent beaucoup plus que l'origine commune : 66,8 % de "Oui" quand ils sont forts contre 31,6 % quand ils sont faibles, alors que l'origine ne change que de 2,8 points (45,0 % contre 42,2 %).
- Les hommes disent "Oui" plus souvent que les femmes : 47,4 % contre 36,5 %.
- Position dans la soirée : 49,9 % de "Oui" au premier rendez-vous, 41,2 % au milieu et 44,5 % au dernier. Être premier donne un petit avantage.

**3 recommandations :**
1. **Mettre en avant les intérêts communs** entre deux profils : c'est le critère qui fait le plus bouger le taux de "Oui".
2. **Soigner la présentation visuelle des profils** (photos), car l'attirance est le critère le plus lié au "Oui".
3. **Laisser de la place à l'humour et à la personnalité** dans les biographies, car l'humour arrive juste derrière.

**Limites à connaître :**
- Ce sont des liens (corrélations), pas des causes.
- Les données viennent d'une expérience de 2002-2004, pas d'utilisateurs de Tinder.
- J'ai supprimé les lignes avec des cases vides (16,1 % des lignes).
- Spark tourne en local : il montre la méthode, pas un vrai gain de vitesse.